# 04 — Cross-task + EVO add-in (standalone)
Run this notebook in **two fresh runtimes at the same time**: one with `TASK = "nuclear"`, one with
`TASK = "small"`. Each exports its Dev OOF early, then waits for the other.
No PDB normalization and no V1/MS retraining: everything is read from caches / saved fold predictions.

In [ ]:
# ============================================================
# BOOTSTRAP — Colab: mount Drive, put pbind on the path
# ============================================================
import sys, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

# Folder of this repo (copy it to Drive once, or git clone it into /content).
REPO = Path("/content/drive/MyDrive/Protein-binding-Phi/protein-binding")
sys.path.insert(0, str(REPO / "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)

# Data root (Drive layout). Override with PBIND_BASE for a local mirror.
# import os; os.environ["PBIND_BASE"] = "/content/drive/MyDrive/Protein-binding-Phi"

from pbind.paths import resolve_paths, check_inputs
display(check_inputs(resolve_paths()))

In [ ]:
TASK = "nuclear"   # or "small"

from pbind.pipelines import open_workspace, features, crosstask
ws = open_workspace(TASK)
features.build_features(ws, attach_only=True)    # cached NA_*/SM_* blocks only

## 1. Export this task's Dev OOF (V1 / MS / Dev-locked ensemble)

In [ ]:
crosstask.export_dev_oof(ws)

## 2. Wait for the counterpart export, attach it, Dev-only screen

In [ ]:
X = crosstask.crosstask_screen(ws, timeout=900)
display(X["cv"]); X["decision"]

## 3. (nuclear only) matched PatchHG vs PatchHG+EVO48, then EVO-PatchHG + MS ensemble

In [ ]:
if TASK == "nuclear":
    E = crosstask.evo_hypergraph_addin(ws)
    E["decision"]